<a href="https://colab.research.google.com/github/collinskiplangatrotich/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duka Electronics: Half-Year Sales Exploration (Jan–Jun 2026)
**Engagement 1 · Executive Sales Review**

## Executive Summary
* **Total Half-Year Performance:** Duka Electronics generated **KES 71,792,500** across **1,000** orders during the January–June 2026 review period.
* **Branch Leadership:** **Nairobi** led all five branches with **KES 26,106,100** in total revenue (**376** orders), accounting for **36.36%** of company-wide sales.
* **Product Mix Contrast:** **Laptops** drove the highest revenue at **KES 41,535,000** (639 units), whereas **Tablets** led unit sales with **691** units sold (generating **KES 19,348,000**).
* **Weekday vs. Weekend Dynamics:** Weekday orders generated **KES 51,856,200** (**723** orders) compared to **KES 19,936,300** (**277** orders) on weekends, with average order values of **KES 71,723.65** and **KES 71,972.20**, respectively.
* **Monthly Peak & High-Value Digital Segment:** Monthly revenue peaked in **2026-04** at **KES 13,518,300**—outperforming the slowest month (**2026-06** at **KES 10,903,400**) by **23.98%**—while Premium online orders from Nairobi and Mombasa contributed **KES 10,218,000** across **83** orders.

In [13]:
import os
import pandas as pd
import numpy as np

os.makedirs("outputs", exist_ok=True)
os.makedirs("excel", exist_ok=True)

# Load dataset with proper date parsing
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("--- Shape ---")
print(df.shape)

print("\n--- Data Types & Non-Null Counts ---")
print(df.info())

print("\n--- Missing Values ---")
print(df.isna().sum())

print("\n--- Date Range ---")
print(f"Start: {df['order_date'].min().strftime('%Y-%m-%d')} | End: {df['order_date'].max().strftime('%Y-%m-%d')}")

print("\n--- Key Categorical Distributions ---")
print("Branches:\n", df["branch"].value_counts(), "\n")
print("Products:\n", df["product"].value_counts(), "\n")
print("Channels:\n", df["channel"].value_counts())

df.head()

--- Shape ---
(1000, 7)

--- Data Types & Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB
None

--- Missing Values ---
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

--- Date Range ---
Start: 2026-01-01 | End: 2026-06-29

--- Key Categorical Distributions ---
Branches:
 branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldo

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


### First-Five-Minutes Inspection Summary
* **Shape & Structure:** The dataset contains **1,000 rows and 7 columns** (`order_id`, `order_date`, `branch`, `product`, `quantity`, `channel`, `unit_price`).
* **Data Types & Date Parsing:** `order_date` was cleanly parsed into `datetime64[ns]`; `order_id`, `quantity`, and `unit_price` are integers (`int64`), and categorical fields (`branch`, `product`, `channel`) are stored as text (`object`).
* **Completeness:** There are **0 missing values** across all 7 columns.
* **Date Range:** Orders span **180 days** across the first half of 2026 (`2026-01-01` to `2026-06-29`).
* **Key Categories:** Five branches (`Nairobi`, `Mombasa`, `Kisumu`, `Nakuru`, `Eldoret`), five products (`Laptop`, `Tablet`, `Phone`, `Headphones`, `Charger`), and three sales channels (`In-store`, `Online`, `Phone order`).

In [14]:
# Create derived features
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.to_period("M").astype(str)
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

df[["order_id", "order_date", "revenue", "month", "weekday", "is_weekend", "price_band"]].head(10)

,order_id,order_date,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,45000,2026-06,Wednesday,False,Value
1,1002,2026-02-02,260000,2026-02,Monday,False,Premium
2,1003,2026-01-05,140000,2026-01,Monday,False,Premium
3,1004,2026-04-26,12500,2026-04,Sunday,True,Value
4,1005,2026-03-07,7500,2026-03,Saturday,True,Value
5,1006,2026-03-26,2500,2026-03,Thursday,False,Value
6,1007,2026-01-15,130000,2026-01,Thursday,False,Premium
7,1008,2026-03-08,6000,2026-03,Sunday,True,Value
8,1009,2026-04-26,12500,2026-04,Sunday,True,Value
9,1010,2026-03-05,6000,2026-03,Thursday,False,Value


In [15]:
## Task 3: Branch Rankings
branch_summary = (
    df.groupby("branch", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        order_count=("order_id", "count"),
        avg_order_value=("revenue", "mean")
    )
    .sort_values(by="total_revenue", ascending=False)
    .reset_index(drop=True)
)
branch_summary["revenue_rank"] = branch_summary["total_revenue"].rank(ascending=False).astype(int)
branch_summary["order_rank"] = branch_summary["order_count"].rank(ascending=False).astype(int)
branch_summary["revenue_share_%"] = (branch_summary["total_revenue"] / branch_summary["total_revenue"].sum() * 100).round(2)

display(branch_summary)

,branch,total_revenue,order_count,avg_order_value,revenue_rank,order_rank,revenue_share_%
0,Nairobi,26106100,376,69431.117021,1,1,36.36
1,Mombasa,14003700,186,75288.709677,2,2,19.51
2,Nakuru,11970400,162,73891.358025,3,3,16.67
3,Kisumu,10868100,159,68352.830189,4,4,15.14
4,Eldoret,8844200,117,75591.452991,5,5,12.32


### Task 3 Answer: Branch Rankings
* **Comparison of Rankings:** Yes, the rankings by total revenue and by order count are **identical** across all five branches: **1. Nairobi** (KES 26,106,100 | 376 orders), **2. Mombasa** (KES 14,003,700 | 186 orders), **3. Nakuru** (KES 11,970,400 | 162 orders), **4. Kisumu** (KES 10,868,100 | 159 orders), and **5. Eldoret** (KES 8,844,200 | 117 orders).
* **Why Rankings Align (and Where Subtle Differences Appear):** Because product selection is uniformly distributed across all branches, order volume is the primary driver of branch ranking. However, slight variations in **average order value (AOV)**—such as Eldoret's higher AOV of **KES 75,591.45** versus Kisumu's **KES 68,352.83**—show how basket size and high-ticket product mix can shift revenue per order even when overall order volume determines the rank order.

In [16]:
## Task 4: Top Products
product_summary = (
    df.groupby("product", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        unit_price=("unit_price", "first")
    )
)

top3_revenue = product_summary.sort_values(by="total_revenue", ascending=False).head(3)
top3_units = product_summary.sort_values(by="units_sold", ascending=False).head(3)

print("--- Top 3 Products by Revenue ---")
display(top3_revenue)

print("\n--- Top 3 Products by Units Sold ---")
display(top3_units)

--- Top 3 Products by Revenue ---


,product,total_revenue,units_sold,unit_price
2,Laptop,41535000,639,65000
4,Tablet,19348000,691,28000
3,Phone,8880000,592,15000



--- Top 3 Products by Units Sold ---


,product,total_revenue,units_sold,unit_price
4,Tablet,19348000,691,28000
2,Laptop,41535000,639,65000
3,Phone,8880000,592,15000


### Task 4 Answer: Product Revenue vs. Unit Volume
* **Top 3 by Revenue:** **1. Laptop** (KES 41,535,000), **2. Tablet** (KES 19,348,000), and **3. Phone** (KES 8,880,000).
* **Top 3 by Units Sold:** **1. Tablet** (691 units), **2. Laptop** (639 units), and **3. Phone** (592 units).
* **Explanation of the Difference:** While the same three core devices occupy the top three spots on both lists, **Tablet** overtakes **Laptop** for #1 in unit volume (+52 more units sold) yet generates less than half of Laptop's revenue. Because a Laptop (KES 65,000) is priced **2.32x higher** than a Tablet (KES 28,000), Laptops easily dominate top-line revenue despite trailing in total units sold.

In [17]:
## Task 5: Weekend vs. Weekday Performance
weekend_summary = (
    df.groupby("is_weekend", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean")
    )
)
weekend_summary["day_type"] = np.where(weekend_summary["is_weekend"], "Weekend (Sat-Sun)", "Weekday (Mon-Fri)")
weekend_summary["avg_order_revenue"] = weekend_summary["avg_order_revenue"].round(2)
display(weekend_summary[["day_type", "order_count", "total_revenue", "avg_order_revenue"]])

,day_type,order_count,total_revenue,avg_order_revenue
0,Weekday (Mon-Fri),723,51856200,71723.65
1,Weekend (Sat-Sun),277,19936300,71972.20


### Task 5 Answer: Weekend vs. Weekday Comparison
* **Volume & Revenue Split:** Weekdays (5 days per week, ~71% of calendar days) naturally generate roughly **2.5x more orders and total revenue** than weekends (2 days per week, ~29% of calendar days).
* **Average Order Revenue:** Average order value remains comparable across both periods, confirming that customer basket composition does not shift dramatically on weekends; rather, total revenue differences are driven primarily by the number of active shopping days.

In [18]:
## Task 6: Monthly Revenue Performance
monthly_rev = (
    df.groupby("month", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("revenue", "sum")
    )
    .sort_values(by="total_revenue", ascending=False)
    .reset_index(drop=True)
)
display(monthly_rev)

best_month = monthly_rev.iloc[0]
worst_month = monthly_rev.iloc[-1]
pct_diff = ((best_month["total_revenue"] - worst_month["total_revenue"]) / worst_month["total_revenue"]) * 100

print(f"Best Month : {best_month['month']} (KES {best_month['total_revenue']:,.0f})")
print(f"Worst Month: {worst_month['month']} (KES {worst_month['total_revenue']:,.0f})")
print(f"Percentage Difference (Best vs. Worst): {pct_diff:.2f}%")

,month,order_count,total_revenue
0,2026-04,167,13518300
1,2026-03,186,12396800
2,2026-01,173,11786600
3,2026-02,159,11772000
4,2026-05,153,11415400
5,2026-06,162,10903400


Best Month : 2026-04 (KES 13,518,300)
Worst Month: 2026-06 (KES 10,903,400)
Percentage Difference (Best vs. Worst): 23.98%


### Task 6 Answer: Best vs. Worst Revenue Month
* **2026-04 (April)** was the best month for revenue at **KES 13,518,300** (167 orders), while **2026-06 (June)** was the lowest-revenue month at **KES 10,903,400** (162 orders), representing a **23.98%** difference ($(\text{13,518,300} - \text{10,903,400}) / \text{10,903,400} \times 100$).


In [19]:
## Task 7: Premium Online Orders in Nairobi & Mombasa
prem_online_coast_capital = df[
    (df["price_band"] == "Premium") &
    (df["channel"] == "Online") &
    (df["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

count_filtered = len(prem_online_coast_capital)
rev_filtered = prem_online_coast_capital["revenue"].sum()

print(f"Matching Orders : {count_filtered}")
print(f"Total Revenue   : KES {rev_filtered:,.0f}")

# Export without index
prem_online_coast_capital.to_csv("outputs/premium_online_coast_capital.csv", index=False)
print("Saved outputs/premium_online_coast_capital.csv successfully.")

Matching Orders : 83
Total Revenue   : KES 10,218,000
Saved outputs/premium_online_coast_capital.csv successfully.


### Task 7 Answer: Premium Online Orders (Nairobi & Mombasa)
* There were **83** Premium (`unit_price >= KES 28,000`) Online orders from Nairobi and Mombasa, generating **KES 10,218,000** in total revenue. The filtered subset has been exported to `outputs/premium_online_coast_capital.csv`.

In [20]:
## Task 8: Cross-Check Table for Excel Spreadsheet (Branch x Channel Pivot)
branch_channel_pivot = pd.pivot_table(
    df,
    values="revenue",
    index="branch",
    columns="channel",
    aggfunc="sum",
    margins=True,
    margins_name="Grand Total"
)
print("--- Reference Pivot Table for Excel Verification ---")
display(branch_channel_pivot)

--- Reference Pivot Table for Excel Verification ---


channel,In-store,Online,Phone order,Grand Total
branch,,,,
Eldoret,4885300,3450700,508200,8844200
Kisumu,6468600,3370800,1028700,10868100
Mombasa,8039600,4086800,1877300,14003700
Nairobi,14808400,7527400,3770300,26106100
Nakuru,6767300,4350400,852700,11970400
Grand Total,40969200,22786100,8037200,71792500


### Task 8 Verification: Spreadsheet Reconciliation
* The `SUMIFS` branch revenue table and the `PivotTable` of revenue by branch and channel in `excel/duka_branch_summary.xlsx` reconcile **100%** with the pandas totals displayed in the reference table above (**Grand Total = KES 71,792,500**).